In [ ]:
## code from Miranda to get the patch/tract info from RA and DEC coords

# Looking for all patches in delta deg region around it
import lsst.geom as geom

dp2_collection = ['LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1',
                  'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2',
                  'LSSTCam/runs/DRP/DP2/v30_0_6_rc1/DM-53881/stage3',
                  'LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4']

skymap =  butler.get('skyMap',
                     skymap='lsst_cells_v2',
                     collections=dp2_collection)

# RA/DEC of interest
ra_bcg = 37.865017
dec_bcg = 6.982205

delta = 0.5
center = geom.SpherePoint(ra_bcg, dec_bcg, geom.degrees)
ra_min, ra_max = ra_bcg - delta, ra_bcg + delta
dec_min, dec_max = dec_bcg - delta, dec_bcg + delta

ra_range = (ra_min, ra_max)
dec_range = (dec_min, dec_max)
# radec = [geom.SpherePoint(ra_range[0], dec_range[0], geom.degrees),
#          geom.SpherePoint(ra_range[0], dec_range[1], geom.degrees),
#          geom.SpherePoint(ra_range[1], dec_range[0], geom.degrees),
#          geom.SpherePoint(ra_range[1], dec_range[1], geom.degrees)]

# this also works with a single RA/DEC coordinate
radec = [geom.SpherePoint(ra_bcg, dec_bcg, geom.degrees)]

tractPatchList = skymap.findTractPatchList(radec)
print(tractPatchList)

# find dataset refs that are within the tract/patch list above
datasetRefs_shear = []

for tractPatch in tractPatchList:
    tract = tractPatch[0]
    patchInfo = tractPatch[1]
    for patch in patchInfo:
        datasetRefs_shear.append(butler.query_datasets('object_shear_all',
                                                 collections=collection,
                                                 tract=tract.tract_id,
                                                 patch=patch.sequential_index))


## code from Miranda to plot a patch image in Firefly

patch = 62

bands = "gri"
images = {}

# get x, y coordinates of object in patch
filt_loc = shear_table_wl_ns['tract'] == tract
filt_loc &= shear_table_wl_ns['patch_y'] == 6
filt_loc &= shear_table_wl_ns['patch_x'] == 2

xys_firefly = []
xys = []

for b in bands:

    coadd = butler.get("deepCoaddCell",
                    tract=tract,
                    patch=patch,
                    band=b,
                    skymap='lsst_cells_v1',
                    collections=[cell_collection, cell_collection_g])

    if b=='g':
        bbox = coadd.outer_bbox
        for i, row in shear_table_wl_ns[filt_loc].iterrows():
            radecs = [row['ra'], row['dec']]
            raDec = geom.SpherePoint(row['ra']*geom.degrees, row['dec']*geom.degrees)
            xy_test = geom.PointI(coadd.wcs.skyToPixel(raDec))
            xys_firefly.append(xy_test)
            xy = xy_test - bbox.getBegin()
            xys.append(xy)

    images[b] = coadd.stitch().asMaskedImage()

    del coadd
    gc.collect()

fig, ax = plt.subplots(1, 1, figsize=(12, 12))

for coord in xys:
    # circle = plt.Circle((coord.getX()-bbox.getBeginX()+50, coord.getY()-bbox.getBeginY()+50), 50, color='cyan', fill=False)
    circle = plt.Circle((coord.getX(), coord.getY()), 50, color='cyan', fill=False)
    ax.add_patch(circle)

rgb = afwRgb.makeRGB(images['i'], images['r'], images['g'], 4, 15, Q=2)
afwRgb.displayRGB(rgb)
plt.show()

import lsst.afw.display as afwDisplay
afwDisplay.setDefaultBackend('firefly')
display1 = afwDisplay.Display(frame=1)
display1.getClient().show_lab_tab

display1.mtv(coadd.stitch().image)
with display1.Buffering():
    for coord in xys_firefly:
        display1.dot('o', coord.getX(), coord.getY(), size=50, ctype='orange')

# display1.erase() # reset the firefly objects (e.g. if you need to update the coordinates)